# cache

> `FeatureCache`: a frozen encoder runs once over the training rows, its outputs are stored as a memory-mapped Arrow dataset, and the head trains at thousands of rows per second; plus the linear head fitted by LBFGS.

In [ ]:
#| default_exp cache

In [ ]:
#| export
import hashlib, json, math, os, shutil, tempfile, time
from pathlib import Path

import numpy as np
import torch
from datasets import Array2D, Dataset, Features, Sequence, Value, load_from_disk
from fastcore.basics import patch, store_attr

from sysone.core import *
from sysone.data import ROW_FEATURES, TypedDecisions
from sysone.models import *
from sysone.losses import soft_ce
from sysone.learner import Learner

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail, test_ne
import logging, sysone
logging.getLogger("transformers").setLevel(logging.ERROR)
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

With the encoder frozen, its output for a row never changes, so running it every epoch is waste. The head-only regime runs it once: `masks` mode stores the vectors at the anchors, k × width per row, which is all a head without layers reads; `rows` mode stores the whole output, for the two-layer head. The cache is an Arrow dataset on disk, memory-mapped when read, keyed by everything the encoder's outputs depend on: the encoder (id, revision and a fingerprint of its weights), the template and transforms, the rows' tokens, and for image rows the images themselves; a second run with the same encoder starts from it and any of these changes rebuilds it. The gold is not part of the key: targets and labels are read from the current rows every time, so correcting a label never costs an encoder pass and never serves the old one. Option shuffling and text augmentation are off while a cache is in use: its rows are static.

How `features` serves a split:

```{mermaid}
flowchart TB
    K["key(split): encoder id, revision and weights fingerprint,<br/>template, transforms, mode, dtype, the rows' tokens (and images)"] --> Q{"on disk?"}
    Q -- no --> E["encode once: eval mode, no gradient,<br/>length-sorted batches within windows of rows"]
    E --> W["save_to_disk: Arrow"]
    W --> R
    Q -- yes --> R["load_from_disk: memory-mapped"]
    R --> G["targets and labels re-attached<br/>from the rows as they are now"]
    G --> D["the dataset the Trainer reads"]
```

And what each mode keeps of one row:

![The encoder's output for one row, and the anchor vectors a masks cache keeps](images/cache_modes.svg)

*Drawn in [figures](90_figures.ipynb#what-a-feature-cache-stores).*

In [ ]:
FIX = "fixtures/tiny-modernbert"
recs = [json.loads(l) for l in Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()]
data = TypedDecisions.from_json("fixtures/typed_decisions_tiny.jsonl", valid=0.25, calib=0.25)
torch.manual_seed(0)
learn = sysone.learner.decision_learner(data, FIX, train="head", head="mlp", preset="cpu", loss="soft_ce")
learn

Learner(fixtures/tiny-modernbert · regime head · head 0 layers/laya · cache None · loss soft_ce · 4,545 of 275,457 params train)

## The key

In [ ]:
#| export
def default_cache_dir() -> Path:
    "`SYSONE_CACHE`, else `~/.cache/sysone/features`"
    return Path(os.environ.get("SYSONE_CACHE", Path.home() / ".cache" / "sysone" / "features"))

def weights_fingerprint(module:torch.nn.Module) -> str:
    "A digest of a module's weights from every tensor's shape, sum and sum of squares (one pass, no copies kept)"
    h = hashlib.sha1()
    with torch.no_grad():
        for n, p in module.named_parameters():
            x = p.detach().float()
            h.update(f"{n}:{tuple(p.shape)}".encode())
            h.update(np.array([x.sum().item(), x.pow(2).sum().item()], dtype=np.float64).tobytes())
    return h.hexdigest()[:16]

def rows_fingerprint(rows) -> str:
    "A digest of the rows' ids, markers and types, and each side stream's ids"
    h = hashlib.sha1()
    names = rows.column_names if hasattr(rows, "column_names") else (list(rows[0]) if len(rows) else [])
    for col in ["input_ids", "markers", "qtype"] + sorted(c for c in names if c.endswith("_input_ids") and c != "input_ids"):
        for v in rows[col] if hasattr(rows, "column_names") else [r[col] for r in rows]: h.update(np.asarray(v, dtype=np.int64).tobytes())
    return h.hexdigest()[:16]

def images_fingerprint(cases, keys=("image",)) -> str:
    "A digest of the images a split's states hold: each file's path, size and modification time, or the inline bytes"
    from sysone.core import _parse_state
    h = hashlib.sha1()
    for i in range(len(cases)):
        st = _parse_state(cases[i]["state"])
        vals = [st.get(k) for k in keys] if isinstance(st, dict) else []
        for v in vals:
            if v is None: continue
            if isinstance(v, str) and not v.startswith("data:") and Path(v).exists():
                f = Path(v).stat(); h.update(f"{v}:{f.st_size}:{f.st_mtime_ns}".encode())
            else: h.update(hashlib.sha1(v if isinstance(v, (bytes, bytearray)) else str(v).encode()).digest())
    return h.hexdigest()[:16]

In [ ]:
enc_fp, rows_fp = weights_fingerprint(learn.model.encoder), rows_fingerprint(data.rows("train"))
test_eq(weights_fingerprint(learn.model.encoder), enc_fp)
test_ne(weights_fingerprint(torch.nn.Linear(3, 3)), enc_fp)
enc_fp, rows_fp

('1497513b9f787829', '672e788b2c9e4af1')

::: {.callout-note title="Finding: a sampled fingerprint misses most weight changes"}
The first `weights_fingerprint` hashed the parameter count and the first 4,096 values of the first, middle and last tensors, which is cheap, and blind to everything else: nudging a norm inside the encoder left the key unchanged, so the cache would have served features of the old weights. It now digests every tensor's name, shape, sum and sum of squares, one pass over the weights with no copies kept:
:::

In [ ]:
def sampled_fingerprint(module):
    "The first version: the parameter count and the first 4,096 values of the first, middle and last tensors"
    ps = list(module.parameters())
    h = hashlib.sha1(str(sum(p.numel() for p in ps)).encode())
    for p in (ps[0], ps[len(ps) // 2], ps[-1]): h.update(p.detach().float().flatten()[:4096].numpy().tobytes())
    return h.hexdigest()[:16]

enc = learn.model.encoder
before = sampled_fingerprint(enc), weights_fingerprint(enc)
enc.embeddings.norm.weight.data += 0.01
after = sampled_fingerprint(enc), weights_fingerprint(enc)
enc.embeddings.norm.weight.data -= 0.01
test_eq(after[0], before[0]); test_ne(after[1], before[1])
{"sampled": (before[0], after[0]), "every tensor": (before[1], after[1])}

{'sampled': ('d4e2c143bd1f6c2e', 'd4e2c143bd1f6c2e'),
 'every tensor': ('1497513b9f787829', 'cfa0b5288824b5ca')}

## FeatureCache

In [ ]:
#| export
class FeatureCache:
    "Encoder outputs at the anchors (`masks`) or at every position (`rows`), computed once, stored as memory-mapped Arrow"
    def __init__(self,
                 model:EncoderDecisionModel, # its encoder is run; it must be frozen
                 data:TypedDecisions, # bound data; the rows to encode
                 mode:str="masks", # 'masks' (k × width per row) or 'rows' (n_tokens × width per row)
                 cache_dir=None, # default_cache_dir() when None
                 batch_size:int=16, # rows per encoder pass
                 dtype:str="float16"): # storage dtype
        if mode not in ("masks", "rows"): raise ValueError(f"mode must be 'masks' or 'rows', not {mode!r}")
        store_attr(but="cache_dir"); self.dir = Path(cache_dir) if cache_dir else default_cache_dir()

    def __repr__(self): return f"FeatureCache({self.mode}, {self.dir})"

In [ ]:
#| export
@patch
def key(self:FeatureCache, split:str) -> str:
    "The cache key of a split: everything the encoder's outputs depend on"
    spec, b = self.model.spec, self.data.builder
    parts = {"encoder": getattr(spec, "id", None), "revision": getattr(spec, "revision", None),
             "weights": weights_fingerprint(self.model.encoder), "template": b.template.hash,
             "tfms": json.dumps([t.to_json() for t in b.tfms], sort_keys=True), "mode": self.mode, "dtype": self.dtype,
             "rows": rows_fingerprint(self.data.rows(split))}
    if b.processor is not None:
        from sysone.template import Image
        parts["images"] = images_fingerprint(self.data.cases[split], tuple({t.key for t in b.tfms if isinstance(t, Image)} or {"image"}))
    hd = self.model.head
    if hd.readout != "anchor" or hd.query is not None or hd.needs_first:     # what the cached vectors are, when not just the anchors
        parts["head"] = {"readout": hd.readout, "query": hd.query, "first": hd.needs_first}
    return hashlib.sha256(json.dumps(parts, sort_keys=True).encode()).hexdigest()[:24]

@patch
def _source(self:FeatureCache, split:str):
    "The rows to encode: static rows, or rows built on read for a processor (images)"
    return self.data.lazy_rows(split) if self.data.builder.processor is not None else self.data.rows(split)

`_encode` runs the encoder in eval mode over the rows and yields each row with its features: its k option vectors (read the way the head reads them, at the anchors or as span means), or the vectors at its n tokens. A head that reads more gets it cached too: a side stream's pooled vector (`query`) and the row's first position (`first`, for an act head). Within a window of rows it batches them by length, so little of each batch is padding, and moves each batch's features off the device in one transfer; rows still come out in their original order. A side stream (a protein that several questions ask about) is encoded once for the whole pass, under `stream_memo`, although its rows fall into different batches. Before the pass, `_prefill_streams` encodes the distinct stream inputs sorted by their own length. A batch of rows sorted by text length can hold proteins of any length, and every one of them would be padded to the longest, so the stream is filled in first, in batches of similar lengths. `features` returns the cached dataset, computing it on the first call:

In [ ]:
#| export
_NOT_EXTRAS = ("input_ids", "attention_mask", "marker_pos", "marker_mask", "qtype", "target", "label", "spans", "answerable")

@patch
def _prefill_streams(self:FeatureCache, rows, dev):
    "Encode each distinct side-stream input of the rows once, in order of length, into the stream memo"
    enc = next((m for m in self.model.encoder.modules() if isinstance(m, StreamEncoder)), None)
    if enc is None or not len(rows): return
    names = rows.column_names if hasattr(rows, "column_names") else list(rows[0])
    for name, side in enc.sides.items():
        col = f"{name}_input_ids"
        if col not in names: continue
        vals = rows[col] if hasattr(rows, "column_names") else [rows[j][col] for j in range(len(rows))]
        todo = sorted({np.asarray(v, dtype=np.int64).tobytes(): v for v in vals}.values(), key=len)
        how = EXTRA_COLLATE.get(col); pad = how[1] if isinstance(how, tuple) else 0
        for i in range(0, len(todo), self.batch_size):
            chunk = todo[i:i + self.batch_size]
            x = torch.full((len(chunk), max(len(v) for v in chunk)), pad, dtype=torch.long); m = torch.zeros_like(x)
            for r, v in enumerate(chunk): x[r, :len(v)] = torch.as_tensor(v); m[r, :len(v)] = 1
            side(x.to(dev), m.to(dev))

@patch
def _encode(self:FeatureCache, rows, window:int|None=None):
    model, pad = self.model, self.data.builder.tok.pad_token_id or 0
    head = model.head
    was_training = model.training; model.eval()
    dev, masks = model.device, self.mode == "masks"
    np_ = lambda t: t.float().cpu().numpy().astype(np_dtype) if t is not None else None
    np_dtype = np.float16 if self.dtype == "float16" else np.float32
    window = window or self.batch_size * 16
    try:
        with torch.no_grad(), stream_memo(model.encoder):     # a side stream shared by several rows is encoded once
            self._prefill_streams(rows, dev)
            for w0 in range(0, len(rows), window):
                items = {j: rows[j] for j in range(w0, min(w0 + window, len(rows)))}
                order, feats = sorted(items, key=lambda j: len(items[j]["input_ids"])), {}
                for i in range(0, len(order), self.batch_size):
                    chunk = order[i:i + self.batch_size]
                    b = collate([items[j] for j in chunk], pad).to(dev)
                    o = model.encode_full(b["input_ids"], b["attention_mask"], **{k: v for k, v in b.items() if k not in _NOT_EXTRAS})
                    h = o.last_hidden_state
                    f = np_(head._readout(h, b["marker_pos"], b.get("spans")) if masks else h)
                    q = np_(model.stream_query(o)) if head.stream_query else None
                    first = np_(h[:, 0]) if masks and head.needs_first else None
                    for r, j in enumerate(chunk):
                        feats[j] = (f[r, :len(items[j]["markers"]) if masks else len(items[j]["input_ids"])],
                                    q[r] if q is not None else None, first[r] if first is not None else None)
                for j, it in items.items():
                    out = {c: it[c] for c in ("input_ids", "markers", "qtype", "target", "label", "answerable", "n_tokens") if c in it}
                    out.update(case=str(it.get("case", "")), qid=str(it.get("qid", "")), case_idx=int(it.get("case_idx", -1)),
                               variant=int(it.get("variant", 0)))
                    out["anchors" if masks else "hidden"], q_, first_ = feats.pop(j)
                    if q_ is not None: out["query"] = q_
                    if first_ is not None: out["first"] = first_
                    yield out
    finally: model.train(was_training)

@patch
def features(self:FeatureCache, split:str="train") -> Dataset:
    "The split's rows with their cached features (computed on first use)"
    path = self.dir / self.key(split)
    if not (path / "dataset_info.json").exists():
        rows, width, hd = self._source(split), self.model.head.width, self.model.head
        col = "anchors" if self.mode == "masks" else "hidden"
        feats = {**{k: v for k, v in ROW_FEATURES.items()}, col: Array2D(shape=(None, width), dtype=self.dtype)}
        if hd.stream_query: feats["query"] = Sequence(Value(self.dtype), length=hd.query_width)
        if self.mode == "masks" and hd.needs_first: feats["first"] = Sequence(Value(self.dtype), length=width)
        feats = Features(feats)
        t0 = time.time()
        tmp = Path(tempfile.mkdtemp(prefix="sysone-cache-"))
        ds = Dataset.from_generator(lambda: self._encode(rows), features=feats, cache_dir=str(tmp), fingerprint=path.name)
        path.parent.mkdir(parents=True, exist_ok=True)
        ds.save_to_disk(str(path))
        shutil.rmtree(tmp, ignore_errors=True)
        (path / "sysone_cache.json").write_text(json.dumps({"split": split, "rows": len(ds), "seconds": round(time.time() - t0, 2),
                                                            "mode": self.mode, "encoder": getattr(self.model.spec, "id", None)}))
    ds = load_from_disk(str(path))
    live = self.data.rows(split)       # the key covers what the encoder saw; the gold, and which case each row is, come from the rows as they are now
    if len(live) == len(ds):
        fresh = [c for c in ("target", "label", "answerable", "case", "qid", "case_idx", "variant") if c in ds.column_names and c in live.column_names]
        ds = ds.remove_columns(fresh)
        for c in fresh: ds = ds.add_column(c, live[c])
    col = "anchors" if self.mode == "masks" else "hidden"
    ds.set_format("numpy", columns=["input_ids", "markers", "target", col] + [c for c in ("query", "first") if c in ds.column_names],
                  output_all_columns=True)
    return ds

In [ ]:
cdir = Path(tempfile.mkdtemp())
fc = FeatureCache(learn.model, data, "masks", cache_dir=cdir)
t0 = time.time(); tr = fc.features("train"); first = time.time() - t0
t0 = time.time(); tr2 = fc.features("train"); second = time.time() - t0
tr, round(first, 3), round(second, 3)

(Dataset({
     features: ['input_ids', 'markers', 'qtype', 'n_tokens', 'anchors', 'target', 'label', 'answerable', 'case', 'qid', 'case_idx', 'variant'],
     num_rows: 55
 }),
 0.114,
 0.042)

In [ ]:
test_eq(len(tr), len(data.rows("train")))
test_eq([list(x) for x in tr["input_ids"]], data.rows("train")["input_ids"])      # length-sorted batches, original order
test_eq(tr[0]["anchors"].shape, (len(tr[0]["markers"]), learn.model.head.width))
test_eq(len(list(cdir.iterdir())), 1)    # the second call re-used the first's files

The cached anchors are exactly what the encoder gives at the markers, so a head without layers scores them as it scores the live encoder's output (up to the float16 storage):

In [ ]:
b = collate([tr[i] for i in range(6)], 3)
live = learn.model(**{k: v for k, v in b.items() if k not in ("anchors", "target", "label")})
cached = learn.model(**{k: v for k, v in b.items() if k not in ("target", "label")})
test_close(cached, live, eps=2e-2)

Changing the template changes the key, and so does training the encoder:

In [ ]:
k0 = fc.key("train")
learn.model.encoder.embeddings.norm.weight.data += 0.01
test_ne(fc.key("train"), k0)
learn.model.encoder.embeddings.norm.weight.data -= 0.01
test_eq(fc.key("train"), k0)

::: {.callout-note title="Finding: the first cache served stale labels"}
The key leaves the gold out on purpose, since correcting a label should not cost an encoder pass. But the first cache also kept the targets it was built with, so a corrected label kept serving the old one. `features` now reads targets and labels from the current rows every time it opens a cache.
:::

::: {.callout-note title="Finding: image rows lost their ids in the cache"}
Rows built on read (image rows, and rows with per-epoch augmentation) came back without the case, question and position the static rows carry, so a cache filled from image rows stored empty ids. Metrics were unaffected, since they read the gold and not the ids. But a per-case look at a multimodal learner's cached predictions found every row under one empty id, and the [screens tutorial](tutorials/web_screens.ipynb)'s first run stopped on it. `LazyRows` now returns the ids with each row. `features` also refreshes them from the current rows along with the gold, which repairs caches filled before the fix.
:::

Changing the gold keeps the key, and the cache serves the new targets:

In [ ]:
has_stop = lambda r: "stop" in r["questions"].get("action", {}).get("criteria", {})
relabel = [dict(r, gold={**r["gold"], "action": {"label": "stop"}}) if has_stop(r) else r for r in recs]
d2 = TypedDecisions.from_records(relabel[:15], valid=relabel[15:], calib=0).bind(learn.data.builder.tok, "laya")
d1 = TypedDecisions.from_records(recs[:15], valid=recs[15:], calib=0).bind(learn.data.builder.tok, "laya")
f1, f2 = FeatureCache(learn.model, d1, "masks", cache_dir=cdir), FeatureCache(learn.model, d2, "masks", cache_dir=cdir)
test_eq(f1.key("train"), f2.key("train"))
old, new = f1.features("train"), f2.features("train")
i = new["qid"].index("action")
test_eq((old["label"][i] != 3, new["label"][i], list(new["target"][i])), (True, 3, [0.0, 0.0, 0.0, 1.0]))

## Training from the cache

A learner made with `cache="masks"` trains from the cached anchors: its datasets are the cached features, and the encoder never runs after the first pass. `cache="rows"` does the same for a head with layers.

In [ ]:
torch.manual_seed(0)
lc = sysone.learner.decision_learner(data, FIX, train="head", cache="masks", preset="cpu", loss="soft_ce", cache_dir=cdir)
t0 = time.time(); lc.fit_one_cycle(3, lr=3e-3); cached_time = time.time() - t0
lc.recorder.metrics[-1]["accuracy"], round(cached_time, 2)

{'eval_loss': '1.191', 'eval_accuracy': '0.2', 'eval_soft_accuracy': '0.3767', 'eval_brier': '0.26', 'eval_kl': '0.4434', 'eval_tv': '0.366', 'eval_ece': '0.124', 'eval_score_mae': '0.838', 'eval_within_one': '0.6', 'eval_n': 25, 'eval_accuracy_choice': '0.1429', 'eval_accuracy_score': '0.2', 'eval_accuracy_noul': '0.25', 'eval_n_unanswerable': 0, 'eval_runtime': '0.0047', 'eval_samples_per_second': '5336', 'eval_steps_per_second': '426.9', 'epoch': '1'}
{'loss': '1.233', 'grad_norm': '0.0003667', 'learning_rate': '0.002714', 'epoch': '1.429'}
{'eval_loss': '1.191', 'eval_accuracy': '0.24', 'eval_soft_accuracy': '0.3767', 'eval_brier': '0.26', 'eval_kl': '0.4434', 'eval_tv': '0.366', 'eval_ece': '0.08404', 'eval_score_mae': '0.838', 'eval_within_one': '0.6', 'eval_n': 25, 'eval_accuracy_choice': '0', 'eval_accuracy_score': '0.4', 'eval_accuracy_noul': '0.25', 'eval_n_unanswerable': 0, 'eval_runtime': '0.0043', 'eval_samples_per_second': '5812', 'eval_steps_per_second': '464.9', 'epoch'

(0.28, 0.37)

In [ ]:
test_eq(lc.model.head.layers, 0)
test_eq("anchors" in lc.dataset("train").column_names, True)
test_eq(len(lc.recorder.metrics), 3)
lr_ = sysone.learner.decision_learner(data, FIX, train="head", cache="rows", preset="cpu", loss="soft_ce", cache_dir=cdir)
lr_.fit(1, lr=1e-3)
test_eq("hidden" in lr_.dataset("train").column_names, True)

{'eval_loss': '1.191', 'eval_accuracy': '0.32', 'eval_soft_accuracy': '0.3767', 'eval_brier': '0.26', 'eval_kl': '0.4434', 'eval_tv': '0.366', 'eval_ece': '0.08401', 'eval_score_mae': '0.838', 'eval_within_one': '0.6', 'eval_n': 25, 'eval_accuracy_choice': '0.1429', 'eval_accuracy_score': '0.2', 'eval_accuracy_noul': '0.625', 'eval_n_unanswerable': 0, 'eval_runtime': '0.0161', 'eval_samples_per_second': '1555', 'eval_steps_per_second': '124.4', 'epoch': '1'}
{'train_runtime': '0.1942', 'train_samples_per_second': '283.1', 'train_steps_per_second': '36.04', 'train_loss': '1.229', 'epoch': '1'}


A head that reads more than the anchors gets it cached beside them: here an act head (the row's first position is cached as `first`) over span readouts, which train from the cache and score it exactly as the live encoder's output:

In [ ]:
torch.manual_seed(0)
la = sysone.learner.decision_learner(data, FIX, train="head", cache="masks", preset="cpu", loss="soft_ce", cache_dir=cdir,
                                     act=True, readout="span", calibrate_after_fit=False)
la.fit(1, lr=1e-3)
ca = la.dataset("train")
test_eq(("first" in ca.column_names, ca[0]["first"].shape), (True, (la.model.head.width,)))
live = FeatureCache(la.model, data, "masks", cache_dir=Path(tempfile.mkdtemp())).key("train")
test_ne(live, FeatureCache(lc.model, data, "masks", cache_dir=cdir).key("train"))          # other vectors, another key
la.model.eval()
b = collate([data.rows("valid")[i] for i in range(4)], data.builder.tok.pad_token_id)
with torch.no_grad():
    lz, az = la.model(**{k: v for k, v in b.items() if k not in ("target", "label")})
    cv = la.dataset("valid")
    cz, caz = la.model(**{k: v for k, v in collate([cv[i] for i in range(4)]).items() if k not in ("target", "label")})
test_close(cz, lz, eps=1e-2); test_close(caz, az, eps=1e-2)
test_fail(lambda: sysone.learner.decision_learner(data, FIX, train="head", cache="masks", head="linear", act=True, preset="cpu",
                                                  cache_dir=cdir).fit_linear(), contains="alone")

{'eval_loss': '1.52', 'eval_accuracy': '0.32', 'eval_soft_accuracy': '0.3753', 'eval_brier': '0.2634', 'eval_kl': '0.4492', 'eval_tv': '0.3658', 'eval_ece': '0.1037', 'eval_score_mae': '0.8417', 'eval_within_one': '0.6', 'eval_n': 25, 'eval_accuracy_choice': '0.1429', 'eval_accuracy_score': '0.2', 'eval_accuracy_noul': '0.625', 'eval_n_unanswerable': 0, 'eval_act_auroc': '0.6985', 'eval_act_brier': '0.2267', 'eval_runtime': '0.0056', 'eval_samples_per_second': '4446', 'eval_steps_per_second': '355.7', 'epoch': '1'}
{'train_runtime': '0.0281', 'train_samples_per_second': '1960', 'train_steps_per_second': '249.4', 'train_loss': '1.6', 'epoch': '1'}


Unfreezing sets the cache aside, since features of a frozen encoder go stale once it trains, and freezing brings it back:

In [ ]:
lc.unfreeze()
test_eq((lc.cache, lc.model.frozen), (None, False))
lc.freeze()
test_eq(lc.cache, "masks")

## The linear head

SetFit's recommended head is a logistic regression fitted on frozen embeddings, in seconds on a CPU. Its counterpart here is `head="linear"`: one weight vector shared by every anchor, $z_i = w^\top h_{m_i} + b$, softmaxed across the row, fitted on all cached training rows at once by LBFGS on the soft cross-entropy, with an L2 penalty, and no backpropagation through anything but the scorer. Features are standardised for the fit and the standardisation is folded back into the weights, so the result is a plain `Linear` scorer. It is the first baseline a run can report, and a check that the cache holds what the head needs.

In [ ]:
#| export
def _stack_anchors(ds):
    X = [np.asarray(a, dtype=np.float32) for a in ds["anchors"]]
    K, D = max(len(x) for x in X), X[0].shape[1]
    A = np.zeros((len(X), K, D), np.float32); M = np.zeros((len(X), K), bool); T = np.zeros((len(X), K), np.float32)
    for i, (x, t) in enumerate(zip(X, ds["target"])): A[i, :len(x)], M[i, :len(x)], T[i, :len(x)] = x, True, t
    return torch.from_numpy(A), torch.from_numpy(M), torch.from_numpy(T)

@patch
def fit_linear(self:Learner, l2:float=1e-3, max_iter:int=200) -> dict:
    "Fit a linear scorer on the cached anchors by LBFGS (needs cache='masks' and head='linear')"
    if self.cache != "masks" or self.model.head.scorer_kind != "linear":
        raise ValueError("fit_linear needs a learner with cache='masks' and head='linear'")
    if self.model.head.act or self.model.head.query is not None:
        raise ValueError("fit_linear fits the linear scorer alone; a head with an act head or a query trains with fit or fit_one_cycle")
    A, M, T = _stack_anchors(self.dataset("train"))
    mu = A[M].mean(0); sd = A[M].std(0).clamp_min(1e-6)
    X = (A - mu) / sd
    w = torch.zeros(X.shape[-1], requires_grad=True)
    opt = torch.optim.LBFGS([w], lr=1.0, max_iter=max_iter, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad()
        loss = soft_ce(X @ w, T, M) + l2 * (w ** 2).sum()
        loss.backward(); return loss
    t0 = time.time(); opt.step(closure)
    with torch.no_grad():
        lin, head = self.model.head.scorer, self.model.head
        if head.standardize:   # the head standardises its input itself: give it these statistics, and the scorer the fitted weights
            head.fit_standardizer(A[M]); lin.weight.copy_(w[None].to(lin.weight.dtype)); lin.bias.fill_(0.0)
        else:                  # fold the standardisation into the weights
            lin.weight.copy_((w / sd)[None].to(lin.weight.dtype)); lin.bias.fill_(float(-(w / sd) @ mu))
        loss = float(soft_ce(X @ w, T, M))
    if self.calibrate_after_fit and self.data.cases.get("calib") is not None: self.calibrate()
    return {"train_loss": loss, "seconds": round(time.time() - t0, 2)}

In [ ]:
torch.manual_seed(0)
llin = sysone.learner.decision_learner(data, FIX, train="head", head="linear", cache="masks", preset="cpu", cache_dir=cdir)
fit = llin.fit_linear()
fit, llin.validate()

({'train_loss': 0.9562579989433289, 'seconds': 0.06},
 {'accuracy': 0.44,
  'soft_accuracy': 0.408012961289173,
  'brier': 0.3349677548258288,
  'kl': 0.684707086938573,
  'tv': 0.3746062292806761,
  'ece': 0.27436781583365943,
  'score_mae': 0.6895297660196107,
  'within_one': 0.7,
  'n': 25,
  'accuracy_choice': 0.5714285714285714,
  'accuracy_score': 0.2,
  'accuracy_noul': 0.625,
  'n_unanswerable': 0})

In [ ]:
from sysone.losses import soft_ce

In [ ]:
test_eq(llin.model.head.scorer_kind, "linear")
A, M, T = _stack_anchors(llin.dataset("train"))
z = llin.model.head.forward_anchors(A, M, torch.zeros(len(A), dtype=torch.long))
test_close(float(soft_ce(z, T, M).detach()), fit["train_loss"], eps=1e-3)
test_fail(lambda: lc.fit_linear(), contains="head='linear'")

A head that standardises its own input (`standardize=True`, the multimodal application's default) takes the fit's statistics itself, and the scorer keeps the weights as fitted; it scores the cached anchors with the same loss:

In [ ]:
torch.manual_seed(0)
lstd = sysone.learner.decision_learner(data, FIX, train="head", head="linear", cache="masks", preset="cpu", cache_dir=cdir, standardize=True)
fit_s = lstd.fit_linear()
test_eq(lstd.model.head.standardizer_fitted, True)
z_s = lstd.model.head.forward_anchors(A, M, torch.zeros(len(A), dtype=torch.long))
test_close(float(soft_ce(z_s, T, M).detach()), fit_s["train_loss"], eps=1e-3)
test_close(fit_s["train_loss"], fit["train_loss"], eps=1e-4)          # the same fit, standardised in the head or folded into it

## What a cache costs

The size of a cache follows from the rows: $\sum_i k_i \times \text{width} \times 2$ bytes in `masks` mode, $\sum_i n_i \times \text{width} \times 2$ bytes in `rows` mode, in float16.

In [ ]:
#| export
def cache_size(rows, width:int, mode:str="masks", bytes_per:int=2) -> int:
    "Bytes a feature cache of `rows` takes"
    n = sum(len(m) for m in rows["markers"]) if mode == "masks" else sum(rows["n_tokens"])
    return n * width * bytes_per

In [ ]:
test_eq(cache_size(data.rows("train"), 64), sum(len(m) for m in data.rows("train")["markers"]) * 128)
{m: f"{cache_size(data.rows('train'), 64, m) / 1e3:.0f} kB" for m in ("masks", "rows")}

{'masks': '25 kB', 'rows': '2114 kB'}

For the real thing, typed-decisions' 6,000 training rows on ModernBERT-large's tokenizer and width:

In [ ]:
#| eval: false
from transformers import AutoTokenizer
td = TypedDecisions.from_hub("LocalLLaMA/typed-decisions", calib=0).bind(AutoTokenizer.from_pretrained("answerdotai/ModernBERT-large"), "laya")
rows = td.rows("train")
{m: f"{cache_size(rows, 1024, m) / 1e6:.0f} MB" for m in ("masks", "rows")}, len(rows)

```
({'masks': '44 MB', 'rows': '3575 MB'}, 6000)
```

That is 3.55 options and 291 tokens per row on average (76 of the 6,000 rows have their state cut at 512 tokens). `masks` mode fits in memory anywhere; `rows` mode is memory-mapped from disk.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()